In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [7]:
videos = ['vid1.mp4', 'vid2.mp4', 'vid3.mp4']

In [8]:
def extract_frames(video_path):

    cap = cv2.VideoCapture(video_path)
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

    cap.release()

    return frames

In [9]:
def compute_background(frames):

    gray_frames = []

    for f in frames:
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        gray_frames.append(gray)

    background = np.median(np.stack(gray_frames), axis=0).astype(np.uint8)

    return background, gray_frames

In [10]:
def extract_foreground(gray_frames, background):

    foreground_frames = []

    for frame in gray_frames:

        diff = cv2.absdiff(frame, background)

        _, fg = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

        foreground_frames.append(fg)

    return foreground_frames

In [11]:
def clean_images(foreground_frames):

    kernel = np.ones((5,5), np.uint8)

    cleaned = []

    for img in foreground_frames:

        c = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)
        c = cv2.morphologyEx(c, cv2.MORPH_CLOSE, kernel)

        cleaned.append(c)

    return cleaned

In [12]:
def detect_edges(clean_frames):

    edges = []

    for img in clean_frames:

        sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

        magnitude = np.sqrt(sobelx**2 + sobely**2)

        magnitude = np.uint8(255 * magnitude / np.max(magnitude))

        edges.append(magnitude)

    return edges

In [13]:
def hough_lines(edge_img):

    height, width = edge_img.shape

    diag_len = int(np.sqrt(height**2 + width**2))

    rhos = np.arange(-diag_len, diag_len, 1)
    thetas = np.deg2rad(np.arange(-90, 90))

    accumulator = np.zeros((len(rhos), len(thetas)), dtype=np.uint64)

    y_idxs, x_idxs = np.nonzero(edge_img)

    for i in range(len(x_idxs)):

        x = x_idxs[i]
        y = y_idxs[i]

        for t_idx in range(len(thetas)):

            rho = int(x*np.cos(thetas[t_idx]) + y*np.sin(thetas[t_idx])) + diag_len

            accumulator[rho, t_idx] += 1

    return accumulator, rhos, thetas

In [14]:
def get_lines(accumulator, rhos, thetas, threshold=150):

    lines = []

    for r in range(accumulator.shape[0]):
        for t in range(accumulator.shape[1]):

            if accumulator[r, t] > threshold:

                rho = rhos[r]
                theta = thetas[t]

                lines.append((rho, theta))

    return lines

In [15]:
def medial_axis(lines):

    if len(lines) < 2:
        return None

    rho_vals = [l[0] for l in lines]
    theta_vals = [l[1] for l in lines]

    rho_mid = np.mean(rho_vals)
    theta_mid = np.mean(theta_vals)

    return rho_mid, theta_mid

In [16]:
for video in videos:

    print("Processing:", video)

    frames = extract_frames(video)

    background, gray_frames = compute_background(frames)

    foreground = extract_foreground(gray_frames, background)

    cleaned = clean_images(foreground)

    edges = detect_edges(cleaned)

    output_frames = []

    for i in range(len(frames)):

        accumulator, rhos, thetas = hough_lines(edges[i])

        lines = get_lines(accumulator, rhos, thetas)

        axis = medial_axis(lines)

        frame = frames[i].copy()

        if axis is not None:

            rho, theta = axis

            a = np.cos(theta)
            b = np.sin(theta)

            x0 = a*rho
            y0 = b*rho

            x1 = int(x0 + 1000*(-b))
            y1 = int(y0 + 1000*(a))

            x2 = int(x0 - 1000*(-b))
            y2 = int(y0 - 1000*(a))

            cv2.line(frame,(x1,y1),(x2,y2),(0,0,255),3)

        output_frames.append(frame)

    height, width, _ = frames[0].shape

    out = cv2.VideoWriter(
        f"output_{video}",
        cv2.VideoWriter_fourcc(*'mp4v'),
        20,
        (width, height)
    )

    for f in output_frames:
        out.write(f)

    out.release()

    print("Saved:", f"output_{video}")

Processing: vid1.mp4
Saved: output_vid1.mp4
Processing: vid2.mp4
Saved: output_vid2.mp4
Processing: vid3.mp4
Saved: output_vid3.mp4
